In [1]:
import os
import boto3
import json
import datetime
from IPython.display import display, JSON
from dotenv import load_dotenv

load_dotenv()
AWS_ACCOUNT_ID = os.environ["AWS_ACCOUNT_ID"]

In [5]:
bedrock_agent = boto3.client(service_name="bedrock-agent", region_name="us-east-1")
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
bedrock_agent_runtime = boto3.client(service_name="bedrock-agent-runtime", region_name="us-east-1")
MODEL_ID = "amazon.nova-micro-v1:0"
MODEL_ID_GPT_OSS_120B= "openai.gpt-oss-120b-1:0"

In [3]:
prompt_name = f"job-description-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

In [4]:
template_text = '''
Você é um assistente de RH.

Escreva uma descrição de vaga profissional e inclusiva usando as informações abaixo:

Cargo: {{job_title}}
Responsabilidades: {{responsibilities}}
Requisitos: {{requirements}}
Local: {{location}}
Regime de trabalho: {{work_type}}

- Comece com um resumo claro
- Use linguagem concisa e inclusiva
- Mantenha o texto com menos de 250 palavras
'''



In [6]:
try:
    response = bedrock_agent.create_prompt(
        name=prompt_name,
        description="Gera descrições de vaga inclusivas a partir de dados estruturados",
        defaultVariant="v1",
        variants=[
            {
                "name": "v1",
                "modelId": MODEL_ID,
                "templateType": "TEXT",
                "templateConfiguration": {
                    "text": {
                        "inputVariables": [
                            {"name": "job_title"},
                            {"name": "responsibilities"},
                            {"name": "requirements"},
                            {"name": "location"},
                            {"name": "work_type"}
                        ],
                        "text": template_text
                    }
                },
                "inferenceConfiguration": {
                    "text": {
                        "maxTokens": 500,
                        "temperature": 0.7,
                        "topP": 0.9,
                        "stopSequences": []
                    }
                }
            }
        ]
    )

except bedrock_agent.exceptions.ConflictException as e:
    print("Já existe!")
    list_response = bedrock_agent.list_prompts()
    response = next((prompt for prompt in list_response['promptSummaries'] if prompt['name'] == prompt_name), None)


print("\n==================== Objeto de resposta ====================\n")
# json.dumps formata o dicionário em uma string JSON bonita
# indent=4 adiciona as quebras de linha e 4 espaços de indentação
# default=str impede que os objetos datetime da AWS quebrem o código
print(json.dumps(response, indent=4, default=str))

print("\n==================== ARN do prompt ====================\n")
print(response['arn'])


==================== Objeto de resposta ====================

{
    "ResponseMetadata": {
        "RequestId": "a84850bd-9214-4312-8d97-b86fd7cbc65b",
        "HTTPStatusCode": 201,
        "HTTPHeaders": {
            "date": "Thu, 22 Oct 2026 19:56:08 GMT",
            "content-type": "application/json",
            "content-length": "1120",
            "connection": "keep-alive",
            "x-amzn-requestid": "a84850bd-9214-4312-8d97-b86fd7cbc65b",
            "x-amz-apigw-id": "FqWN5HXuIAMESiA=",
            "x-amzn-trace-id": "Root=1-6ada6a58-437bad1e39c6ccdc2942e699"
        },
        "RetryAttempts": 0
    },
    "name": "job-description-20261022165333",
    "description": "Gera descri\u00e7\u00f5es de vaga inclusivas a partir de dados estruturados",
    "defaultVariant": "v1",
    "variants": [
        {
            "name": "v1",
            "templateType": "TEXT",
            "templateConfiguration": {
                "text": {
                    "text": "\nVoc\u00ea \u00

## Criando uma nova versão do prompt

In [8]:
response = bedrock_agent.create_prompt_version(
    description='Prompt inicial para criar descrições de vagas.',
    promptIdentifier=f"arn:aws:bedrock:us-east-1:{AWS_ACCOUNT_ID}:prompt/1TGYIQM8WU"
)

print("\n==================== ARN DA VERSÃO DO PROMPT ====================\n")
print(response["arn"])


==================== ARN DA VERSÃO DO PROMPT ====================

arn:aws:bedrock:us-east-1:<ACCOUNT_ID>:prompt/1TGYIQM8WU:1


## Invocando o modelo com a Converse API

In [10]:
prompt_arn_with_version = f"arn:aws:bedrock:us-east-1:{AWS_ACCOUNT_ID}:prompt/1TGYIQM8WU:1"

response = bedrock_runtime.converse(
    modelId=prompt_arn_with_version,
    promptVariables={
        "job_title": {"text": "Designer de UX"},
        "responsibilities": {"text": "Projetar interfaces de usuário, conduzir testes de usabilidade, colaborar com os times de produto"},
        "requirements": {"text": "3+ anos de experiência, Figma, conhecimento de HTML/CSS, boa comunicação"},
        "location": {"text": "Nova York ou remoto"},
        "work_type": {"text": "Tempo integral"}
    }
)

print("\n==================== Texto da resposta ====================\n")
print(response['output']['message']['content'][0]['text'])


==================== Texto da resposta ====================

**Vaga: Designer de UX**

Estamos procurando um Designer de UX talentoso para se juntar à nossa equipe em Nova York ou remotamente. Como Designer de UX, você será responsável por projetar interfaces de usuário inovadoras e intuitivas que melhorem a experiência do usuário em nossos produtos. Este cargo é ideal para alguém que adora colaborar e tem um forte foco em usabilidade.

**Responsabilidades:**
- Desenvolver interfaces de usuário que sejam acessíveis e intuitivas.
- Realizar testes de usabilidade para garantir que os designs atendam às necessidades dos usuários.
- Colaborar estreitamente com os times de produto para integrar feedback e iterações.

**Requisitos:**
- Mínimo de 3 anos de experiência em design de UX.
- Proficiência em Figma.
- Conhecimento de HTML/CSS para implementar designs.
- Excelentes habilidades de comunicação e colaboração.

**Local:** Nova York ou remoto.

**Regime de trabalho:** Tempo integral.

Se

## Otimizando o prompt

In [11]:
def handle_response_stream(response):
    try:
        event_stream = response['optimizedPrompt']
        for event in event_stream:
            if 'optimizedPromptEvent' in event:
                print("\n==================== PROMPT OTIMIZADO ====================\n")
                optimized_prompt = event['optimizedPromptEvent']['optimizedPrompt']['textPrompt']['text']
                return optimized_prompt 
    except Exception as e:
        raise e


prompt_input = {
    "textPrompt": {
        "text": template_text
    }
}

response = bedrock_agent_runtime.optimize_prompt(
            input=prompt_input,
            targetModelId=MODEL_ID
        )

print("\n==================== PROMPT ORIGINAL ====================\n")
print(template_text)
optimized_prompt = handle_response_stream(response)
print(optimized_prompt)


==================== PROMPT ORIGINAL ====================


Você é um assistente de RH.

Escreva uma descrição de vaga profissional e inclusiva usando as informações abaixo:

Cargo: {{job_title}}
Responsabilidades: {{responsibilities}}
Requisitos: {{requirements}}
Local: {{location}}
Regime de trabalho: {{work_type}}

- Comece com um resumo claro
- Use linguagem concisa e inclusiva
- Mantenha o texto com menos de 250 palavras


==================== PROMPT OTIMIZADO ====================

"## Instruction\n\nYou are an HR assistant specialized in creating professional and inclusive job descriptions. Your task is to write a compelling job posting that attracts diverse candidates while clearly communicating the role's requirements and responsibilities.\n\n## Job Information\n\n### Position Title\n{{job_title}}\n\n### Key Responsibilities\n{{responsibilities}}\n\n### Requirements\n{{requirements}}\n\n### Location\n{{location}}\n\n### Work Arrangement\n{{work_type}}\n\n## Guidelines for Writ

In [12]:
prompt_identifier = f"arn:aws:bedrock:us-east-1:{AWS_ACCOUNT_ID}:prompt/1TGYIQM8WU"

existing_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_identifier)
print("\n==================== PROMPT ORIGINAL ====================\n")
print(existing_prompt["variants"][0]["templateConfiguration"]["text"]["text"])


updated_variants = existing_prompt["variants"]
for variant in updated_variants:
    if variant["templateType"] == "TEXT":
        variant["templateConfiguration"]["text"]["text"] = optimized_prompt


response = bedrock_agent.update_prompt(
    promptIdentifier=prompt_identifier,
    name=existing_prompt["name"],
    description=existing_prompt["description"],
    defaultVariant=existing_prompt["defaultVariant"],
    variants=updated_variants
)

print("\n==================== PROMPT ATUALIZADO ====================\n")

updated_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_identifier)
print(updated_prompt["variants"][0]["templateConfiguration"]["text"]["text"])


==================== PROMPT ORIGINAL ====================


Você é um assistente de RH.

Escreva uma descrição de vaga profissional e inclusiva usando as informações abaixo:

Cargo: {{job_title}}
Responsabilidades: {{responsibilities}}
Requisitos: {{requirements}}
Local: {{location}}
Regime de trabalho: {{work_type}}

- Comece com um resumo claro
- Use linguagem concisa e inclusiva
- Mantenha o texto com menos de 250 palavras


==================== PROMPT ATUALIZADO ====================

"## Instruction\n\nYou are an HR assistant specialized in creating professional and inclusive job descriptions. Your task is to write a compelling job posting that attracts diverse candidates while clearly communicating the role's requirements and responsibilities.\n\n## Job Information\n\n### Position Title\n{{job_title}}\n\n### Key Responsibilities\n{{responsibilities}}\n\n### Requirements\n{{requirements}}\n\n### Location\n{{location}}\n\n### Work Arrangement\n{{work_type}}\n\n## Guidelines for Wri

In [13]:
response = bedrock_agent.create_prompt_version(
    description='Prompt otimizado para criar descrições de vagas.',
    promptIdentifier=prompt_identifier
)

print("\n==================== ARN DA VERSÃO DO PROMPT ====================\n")
print(response["arn"])


==================== ARN DA VERSÃO DO PROMPT ====================

arn:aws:bedrock:us-east-1:<ACCOUNT_ID>:prompt/1TGYIQM8WU:2


## Comparando versões

In [14]:
prompt_arn = f"arn:aws:bedrock:us-east-1:{AWS_ACCOUNT_ID}:prompt/1TGYIQM8WU:1"

response = bedrock_runtime.converse(
    modelId=prompt_arn,
    promptVariables={
        "job_title": {"text": "Designer de UX"},
        "responsibilities": {"text": "Projetar interfaces de usuário, conduzir testes de usabilidade, colaborar com os times de produto"},
        "requirements": {"text": "3+ anos de experiência, Figma, conhecimento de HTML/CSS, boa comunicação"},
        "location": {"text": "Nova York ou remoto"},
        "work_type": {"text": "Tempo integral"}
    }
)

print("\\n==================== Texto da resposta ====================\\n")
print(response['output']['message']['content'][0]['text'])

\n==================== Texto da resposta ====================\n
**Descrição de Vaga: Designer de UX**

Estamos procurando um Designer de UX inovador e experiente para se juntar à nossa equipe. Como Designer de UX, você será responsável por projetar interfaces de usuário intuitivas e acessíveis que melhorem a experiência do usuário em nossos produtos. Seu trabalho será crucial para garantir que nossos produtos sejam não apenas funcionais, mas também agradáveis e fáceis de usar.

**Responsabilidades:**
- Desenvolver interfaces de usuário inovadoras e acessíveis.
- Realizar testes de usabilidade para garantir a qualidade e a eficácia do design.
- Colaborar estreitamente com os times de produto para entender as necessidades dos usuários e alinhar as estratégias de design.

**Requisitos:**
- Experiência de 3+ anos em design de UX.
- Proficiência em ferramentas como Figma.
- Conhecimento em HTML/CSS.
- Excelentes habilidades de comunicação e colaboração.

**Local:** Nova York ou remoto

**Re

In [15]:
prompt_arn = f"arn:aws:bedrock:us-east-1:{AWS_ACCOUNT_ID}:prompt/1TGYIQM8WU:2"

response = bedrock_runtime.converse(
    modelId=prompt_arn,
    promptVariables={
        "job_title": {"text": "Designer de UX"},
        "responsibilities": {"text": "Projetar interfaces de usuário, conduzir testes de usabilidade, colaborar com os times de produto"},
        "requirements": {"text": "3+ anos de experiência, Figma, conhecimento de HTML/CSS, boa comunicação"},
        "location": {"text": "Nova York ou remoto"},
        "work_type": {"text": "Tempo integral"}
    }
)

print("\\n==================== Texto da resposta ====================\\n")
print(response['output']['message']['content'][0]['text'])

\n==================== Texto da resposta ====================\n
**Designer de UX**

Você é responsável por projetar interfaces de usuário intuitivas, conduzir testes de usabilidade e colaborar com os times de produto para melhorar a experiência do usuário. Se tem mais de 3 anos de experiência, conhecimento em Figma, HTML/CSS e uma boa capacidade de comunicação, essa pode ser a oportunidade perfeita para você. O trabalho pode ser realizado em Nova York ou remotamente em tempo integral.

**Responsabilidades principais:**
- Projeto de interfaces de usuário.
- Condução de testes de usabilidade.
- Colaboração com times de produto.

**Requisitos:**
- 3+ anos de experiência.
- Familiaridade com Figma.
- Conhecimento de HTML/CSS.
- Boa comunicação.

**Localização e Horário:**
- Nova York ou remoto.
- Tempo integral.


In [ ]:
}